In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt, resample_poly

def clean_single_signal(sig: np.ndarray) -> np.ndarray:

    """
    Clean a single ECG signal to save RAM.
    Detect and handle:
      • NaN / Inf values      → interpolate per lead
      • Completely flat leads → return None (mark as corrupted)
      • Amplitude clipping    → clip to ±5 mV (physiological range)
    """
    T, C = sig.shape
    
    for c in range(C):
        lead = sig[:, c]
        bad  = ~np.isfinite(lead)
        if bad.any():
            idx  = np.arange(T)
            good = np.where(~bad)[0]
            if len(good) < 2:               
                return None # الإشارة كلها تالفة
            sig[:, c] = np.interp(idx, good, lead[good]) #تقريب خطي

    if (sig.std(axis=0) < 1e-6).any():
        return None

    sig = np.clip(sig, -5.0, 5.0)

    return sig

In [ ]:
def bandpass_filter_single(sig: np.ndarray,
                           fs: int = 500, 
                           low: float = 0.5, high: float = 40.0, 
                           order: int = 4) -> np.ndarray:
    
    nyq = fs / 2.0  #(Nyquist) تردد نصف العينة ، كل تردد بنقطتين
    sos = butter(order, [low / nyq, high / nyq], btype="band", output="sos") #(Butterworth) soft filter coefficients
    #Second-Order Sections 
    sig_f = np.empty_like(sig)
    
    C = sig.shape[1] # عدد المسارات (12)
    # تطبيق الفلتر على كل سلك لوحده
    for c in range(C):
        sig_f[:, c] = sosfiltfilt(sos, sig[:, c])  #(Forward and Backward)zero-phase shift
        
    return sig_f


In [6]:
def resample_single_signal(sig: np.ndarray,
                           orig_fs: int,
                           target_fs: int) -> np.ndarray:

    if orig_fs == target_fs:
        return sig
    
    sig_r = resample_poly(sig, target_fs, orig_fs, axis=0).astype(np.float32) #completing the axis points to 500

    return sig_r



In [ ]:
def normalize_single_signal(sig: np.ndarray, method: str = "zscore") -> np.ndarray:
    
    sig_n = sig.copy()
    
    if method == "zscore":
        # حساب المتوسط والانحراف المعياري لكل سلك
        mu  = sig_n.mean(axis=0, keepdims=True)
        std = sig_n.std(axis=0, keepdims=True) + 1e-8  # no division by zero
        sig_n = (sig_n - mu) / std
        sig_n = np.clip(sig_n, -5.0, 5.0)  # remove outliers after zscore
        
    elif method == "minmax":
        mn  = sig_n.min(axis=0, keepdims=True)
        mx  = sig_n.max(axis=0, keepdims=True)
        sig_n = (sig_n - mn) / (mx - mn + 1e-8)  # result is always 0 → 1
        
    return sig_n.astype(np.float32)